In [ ]:
# Cell 0: intro text only (plain print - never paste markdown tables into code cells)
print("GPU: count isolated 1s in a binary matrix (Colab / CUDA)")
print("Island = cell with 1 where all 8 neighbors (incl. diagonals) are 0.")
print("Boundary counts as 0.")
print()
print("Methods:")
print("  1  interview kernel: one if (&&), global atomic counter++")
print("  1b divergent early-return + global atomic (same class, diff branches)")
print("  2  uniform path + padded grid + block reduce")
print("  3  compaction (flatnonzero) + sparse kernel")
print("  4  warp-local reduce in shared mem (ballot N/A in numba_cuda)")
print("  5  padded grid, 30x30 tile, 16x16 launch (256 thr)")
print("  5d 30x30 tile == 30x30 block (900 thr; may OOR on Colab T4)")
print("  6_lut_k3 k=3 bitmask LUT, warp/tile (offline table)")
print()
print("Benchmark N: 1024 and 2048.")
print("Colab: Runtime -> T4 GPU -> Restart session -> run cell 1 (or cell 1b if GPU quota exceeded).")

## Interview question (original)

```cpp
// One thread per cell (i, j). On GPU, counter++  =>  atomicAdd(&counter, 1).
if (mat[i][j] == 1
    && mat[i-1][j-1] == 0 && mat[i-1][j] == 0 && mat[i-1][j+1] == 0
    && mat[i][j-1] == 0                   && mat[i][j+1] == 0
    && mat[i+1][j-1] == 0 && mat[i+1][j] == 0 && mat[i+1][j+1] == 0)
    counter++;
```

**Why inefficient?** (1) **Warp divergence** — lanes take different paths through the `&&` chain. (2) **Global atomic** on one counter — serializes hits. (3) **Redundant global loads** — no shared/LDS tile. (4) **Boundary branches** in the `&&` (each neighbor needs `i`/`j` edge guards — e.g. up-left needs `i==0 or j==0 or ...`, not just `i==0`).

**Notebook mapping**

| Key | What it is |
|-----|------------|
| `1_interview_if_atomic` | Literal `if (&& …)` + `atomicAdd` — the interview kernel |
| `1b_divergent_early_return` | Same logic, **early return** per check — still divergent + atomic, slightly different branch structure |
| `2`–`5` | Fixes: uniform + block reduce, compaction, tile in LDS, etc. |

Methods **1** and **1b** should give **the same count**; timings may differ a little.

In [ ]:
# ========== START HERE (cell 1): imports + GPU check ==========
# Colab: Runtime -> Change runtime type -> T4 GPU -> Save -> Restart session
import os
import sys
import time
import subprocess
import warnings
import numpy as np

_GPU_HELP = (
    "No CUDA GPU in this session.\n\n"
    "If Colab says 'Cannot connect to GPU backend' / usage limits:\n"
    "  - Free-tier GPU quota is exhausted; wait (often resets daily) or use Colab Pro\n"
    "  - Run **cell 1b** instead for CPU simulator (correctness only, not real timings)\n"
    "  - Or try Kaggle Notebooks (30h GPU/week) / local NVIDIA GPU\n\n"
    "Otherwise (CPU runtime selected by mistake):\n"
    "  1. Runtime -> Change runtime type -> T4 GPU -> Save\n"
    "  2. Runtime -> Restart session\n"
    "  3. Re-run cells 0 and 1"
)

# --- diagnose BEFORE numba loads the CUDA driver ---
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    print(smi.stderr or smi.stdout or "nvidia-smi not found")
    raise RuntimeError(_GPU_HELP)
print("nvidia-smi:", smi.stdout.splitlines()[2].strip())

for _libcuda in (
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
):
    if os.path.isfile(_libcuda):
        os.environ.setdefault("NUMBA_CUDA_DRIVER", _libcuda)
        print("NUMBA_CUDA_DRIVER =", os.environ["NUMBA_CUDA_DRIVER"])
        break

try:
    from numba import cuda
    import numba
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
    from numba import cuda
    import numba

try:
    import cupy as cp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
    import cupy as cp

print("numba", numba.__version__)
print("cupy", cp.__version__)
warnings.filterwarnings("ignore", message=".*Grid size.*under-utilization.*")

try:
    if not cuda.is_available():
        raise RuntimeError(_GPU_HELP)
    cuda.select_device(0)
    dev = cuda.get_current_device()
except Exception as e:
    raise RuntimeError(_GPU_HELP + "\n\nOriginal error: " + str(e)) from e

print("CUDA device:", dev.name)
print("compute capability:", dev.compute_capability)
CUDA_SIMULATOR = False
print("GPU OK — continue with the cells below.")

## Cell 1b: no GPU? (Colab quota / CPU runtime)

Run **instead of cell 1** when Colab shows *Cannot connect to GPU backend*.

- Numba **CUDA simulator** on CPU — limited smoke test only
- **GPU kernels do not run** in numba 0.60 simulator (`cuda.atomic` and `dev[i]=` both RecursionError)
- Correctness cell runs **CPU-only** tile/LUT checks; use **cell 1 + real GPU** for full validation
- For full checks + benchmark: re-run **cell 1** on a real GPU once quota returns

In [ ]:
# ========== cell 1b: CPU simulator (skip cell 1) ==========
import os
os.environ["NUMBA_ENABLE_CUDASIM"] = "1"  # must be set before cuda import

import sys
import time
import subprocess
import warnings
import numpy as np

try:
    from numba import cuda
    import numba
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
    from numba import cuda
    import numba

cp = None
try:
    import cupy as cp
except ImportError:
    pass

print("MODE: CPU CUDA simulator — correctness only, NOT real GPU timings")
print("numba", numba.__version__)
print("cupy", cp.__version__ if cp is not None else "not loaded")
warnings.filterwarnings("ignore", message=".*Grid size.*under-utilization.*")
cuda.select_device(0)
CUDA_SIMULATOR = True
print("Simulator OK — CPU-only correctness; do NOT run GPU benchmark.")

## Reference (CPU) and test data

In [ ]:
def count_isolated_cpu(grid: np.ndarray) -> int:
    """8-neighbor isolation. grid: uint8, shape (N, N)."""
    N = grid.shape[0]
    c = 0
    for i in range(N):
        for j in range(N):
            if grid[i, j] != 1:
                continue
            has_neighbor_one = False
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0:
                        continue
                    ni, nj = i + di, j + dj
                    if 0 <= ni < N and 0 <= nj < N and grid[ni, nj] == 1:
                        has_neighbor_one = True
                        break
                if has_neighbor_one:
                    break
            if not has_neighbor_one:
                c += 1
    return c


def make_grid(N: int, density: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    g = (rng.random((N, N)) < density).astype(np.uint8)
    return g


def pad_grid(grid: np.ndarray) -> np.ndarray:
    """(N+2)x(N+2) with a zero border; interior is grid."""
    N = grid.shape[0]
    p = np.zeros((N + 2, N + 2), dtype=np.uint8)
    p[1 : N + 1, 1 : N + 1] = grid
    return p


def pack_row_word(grid: np.ndarray, r: int, col_start: int) -> int:
    N = grid.shape[0]
    w = 0
    for b in range(32):
        c = col_start + b
        if c < N and grid[r, c]:
            w |= 1 << b
    return w


def pack_row_word_padded(padded: np.ndarray, r: int, col_start: int, N: int) -> int:
    """Pack 32 cols from interior row r (0..N-1) using padded (N+2)x(N+2)."""
    w = 0
    for b in range(32):
        c = col_start + b
        if c < N and padded[r + 1, c + 1]:
            w |= 1 << b
    return w


def count_stripe8_bitwise_cpu(padded: np.ndarray, N: int) -> int:
    """Method 5 CPU: same as GPU - 8-neighbor check on padded grid (all rows/cols)."""
    c = 0
    for r in range(N):
        for col in range(N):
            if padded[r + 1, col + 1] != 1:
                continue
            ok = True
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    if di == 0 and dj == 0:
                        continue
                    if padded[r + 1 + di, col + 1 + dj] == 1:
                        ok = False
            if ok:
                c += 1
    return c


# quick sanity (8-neighbor islands)
toy = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=np.uint8)
assert count_isolated_cpu(toy) == 1
pair = np.array([[0, 1, 0], [0, 1, 0], [0, 0, 0]], dtype=np.uint8)
assert count_isolated_cpu(pair) == 0
diag = np.array([[1, 0], [0, 1]], dtype=np.uint8)
assert count_isolated_cpu(diag) == 0
assert count_stripe8_bitwise_cpu(pad_grid(toy), 3) == count_isolated_cpu(toy)
assert count_stripe8_bitwise_cpu(pad_grid(diag), 2) == count_isolated_cpu(diag)
print("CPU reference OK (all methods, incl. bitwise stripe-5)")

## CUDA kernels (Numba)

## Method 6: k=3 tile LUT (benchmarked)

**6_lut_k3** — k=3 **2^25 LUT** (~32 MB), warp/tile `(32,1)`, raw `lut[bits]`.

Older stencil variants (6/6a/6b) removed from benchmark; **6_lut_k3** was fastest in A/B.

In [ ]:
TILE_K3 = 3
TILE_K3_HALO = TILE_K3 + 2  # 5
TILE_K3_LUT_BITS = TILE_K3_HALO * TILE_K3_HALO  # 25
TILE_K3_LUT_SIZE = 1 << TILE_K3_LUT_BITS
TILES_PER_WARP = 8  # legacy 6a serial (kernel kept, not benchmarked)

BIT_TILE_K = 4  # 6 k=4 stencil (not benchmarked)
BIT_TILE_HALO = BIT_TILE_K + 2


def _count_tile_interior_sm(sm: np.ndarray, k: int, halo: int) -> int:
    """Count islands in interior k×k from flat halo sm[ly*halo+lx]."""
    c = 0
    for i in range(1, k + 1):
        for j in range(1, k + 1):
            si = i * halo + j
            if sm[si] != 1:
                continue
            ok = True
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0:
                        continue
                    if sm[(i + di) * halo + (j + dj)] != 0:
                        ok = False
                        break
                if not ok:
                    break
            if ok:
                c += 1
    return c


def count_isolated_tile_cpu(padded: np.ndarray, N: int, k: int) -> int:
    halo = k + 2
    nrow, ncol = padded.shape
    bx = (N + k - 1) // k
    total = 0
    for by in range(bx):
        for bxi in range(bx):
            tr, tc = by * k, bxi * k
            sm = np.zeros(halo * halo, dtype=np.uint8)
            for i in range(halo):
                for j in range(halo):
                    sr, sc = tr + i, tc + j
                    if sr < nrow and sc < ncol:
                        sm[i * halo + j] = padded[sr, sc]
            total += _count_tile_interior_sm(sm, k, halo)
    return total


@numba.njit(cache=True)
def _count_from_bits(bits, k, halo):
    c = 0
    for i in range(1, k + 1):
        for j in range(1, k + 1):
            si = i * halo + j
            if ((bits >> si) & 1) == 0:
                continue
            ok = True
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    if di == 0 and dj == 0:
                        continue
                    if ((bits >> ((i + di) * halo + (j + dj))) & 1) != 0:
                        ok = False
                        break
                if not ok:
                    break
            if ok:
                c += 1
    return c


@numba.njit(parallel=True, cache=True)
def _build_lut_k3(k, halo, n_total):
    lut = np.zeros(n_total, dtype=np.uint8)
    for bits in numba.prange(n_total):
        lut[bits] = _count_from_bits(bits, k, halo)
    return lut


def build_tile_k3_lut():
    k, halo = TILE_K3, TILE_K3_HALO
    n_total = TILE_K3_LUT_SIZE
    t0 = time.perf_counter()
    lut = _build_lut_k3(k, halo, n_total)
    elapsed = time.perf_counter() - t0
    return {"k": k, "halo": halo, "lut": lut, "build_s": elapsed}


TILE_K3_TABLE = build_tile_k3_lut()
TILE_K3_LUT = TILE_K3_TABLE["lut"]
print(
    f"k=3 LUT: 2^{TILE_K3_LUT_BITS} entries, "
    f"{TILE_K3_LUT.nbytes / 1e6:.1f} MB, built in {TILE_K3_TABLE['build_s']:.2f}s"
)

for N, dens in ((16, 0.15), (32, 0.08), (64, 0.05)):
    h = make_grid(N, dens, seed=100 + N)
    p = pad_grid(h)
    ref = count_isolated_cpu(h)
    got = count_isolated_tile_cpu(p, N, TILE_K3)
    assert got == ref, (N, got, ref)
print("tile k=3 stencil CPU sum OK")

In [ ]:
TPB = 16  # threads per block dim (16x16 = 256 threads); must match BLOCK_THREADS
BLOCK_THREADS = 256  # literal: required by numba_cuda for cuda.shared.array()
WARPS_PER_BLOCK = 8  # 256 // 32
WARP = 32


@cuda.jit
def kernel_interview_if_atomic(grid, N, out):
    """Interview baseline: if (mat[i][j]==1 && all 8 nbrs==0) counter++ (atomicAdd)."""
    i = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    j = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i >= N or j >= N:
        return
    if (
        grid[i, j] == 1
        and (i == 0 or j == 0 or grid[i - 1, j - 1] == 0)
        and (i == 0 or grid[i - 1, j] == 0)
        and (i == 0 or j == N - 1 or grid[i - 1, j + 1] == 0)
        and (j == 0 or grid[i, j - 1] == 0)
        and (j == N - 1 or grid[i, j + 1] == 0)
        and (i == N - 1 or j == 0 or grid[i + 1, j - 1] == 0)
        and (i == N - 1 or grid[i + 1, j] == 0)
        and (i == N - 1 or j == N - 1 or grid[i + 1, j + 1] == 0)
    ):
        cuda.atomic.add(out, 0, 1)


@cuda.jit
def kernel_divergent(grid, N, out):
    """Method 1b: same test, early return per check (also divergent + atomic)."""
    i = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    j = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i >= N or j >= N:
        return
    if grid[i, j] != 1:
        return
    if i > 0 and grid[i - 1, j] == 1:
        return
    if i < N - 1 and grid[i + 1, j] == 1:
        return
    if j > 0 and grid[i, j - 1] == 1:
        return
    if j < N - 1 and grid[i, j + 1] == 1:
        return
    if i > 0 and j > 0 and grid[i - 1, j - 1] == 1:
        return
    if i > 0 and j < N - 1 and grid[i - 1, j + 1] == 1:
        return
    if i < N - 1 and j > 0 and grid[i + 1, j - 1] == 1:
        return
    if i < N - 1 and j < N - 1 and grid[i + 1, j + 1] == 1:
        return
    cuda.atomic.add(out, 0, 1)


@cuda.jit
def kernel_uniform_padded(padded, N, block_out):
    """Method 2: same control flow for all threads; per-block count in block_out[blockIdx]."""
    # interior coordinates in padded array
    i = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y + 1
    j = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x + 1
    if i > N or j > N:
        return

    v = padded[i, j]
    n = (
        padded[i - 1, j - 1] + padded[i - 1, j] + padded[i - 1, j + 1]
        + padded[i, j - 1] + padded[i, j + 1]
        + padded[i + 1, j - 1] + padded[i + 1, j] + padded[i + 1, j + 1]
    )
    isolated = 1 if (v == 1 and n == 0) else 0

    tid = cuda.threadIdx.x + cuda.threadIdx.y * cuda.blockDim.x
    sm = cuda.shared.array(256, dtype=numba.int32)
    sm[tid] = isolated
    cuda.syncthreads()

    stride = 128
    while stride > 0:
        if tid < stride:
            sm[tid] += sm[tid + stride]
        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = sm[0]


@cuda.jit
def kernel_compact_check(padded, N, positions, num_ones, out):
    """Method 3: one thread per '1' cell (compacted index list)."""
    k = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if k >= num_ones:
        return
    idx = positions[k]
    j = idx % N
    i = idx // N
    pi, pj = i + 1, j + 1
    n = (
        padded[pi - 1, pj - 1] + padded[pi - 1, pj] + padded[pi - 1, pj + 1]
        + padded[pi, pj - 1] + padded[pi, pj + 1]
        + padded[pi + 1, pj - 1] + padded[pi + 1, pj] + padded[pi + 1, pj + 1]
    )
    if n == 0:
        cuda.atomic.add(out, 0, 1)


@cuda.jit
def kernel_ballot_padded(padded, N, block_out):
    """Method 4: uniform path + full 256-thread block reduce (same as method 2)."""
    i = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y + 1
    j = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x + 1
    if i > N or j > N:
        isolated = 0
    else:
        v = padded[i, j]
        n = (
            padded[i - 1, j - 1] + padded[i - 1, j] + padded[i - 1, j + 1]
            + padded[i, j - 1] + padded[i, j + 1]
            + padded[i + 1, j - 1] + padded[i + 1, j] + padded[i + 1, j + 1]
        )
        isolated = 1 if (v == 1 and n == 0) else 0

    tid = cuda.threadIdx.x + cuda.threadIdx.y * cuda.blockDim.x
    sm = cuda.shared.array(256, dtype=numba.int32)
    sm[tid] = isolated
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if tid < stride:
            sm[tid] += sm[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = sm[0]


TILE = 30
HALO = 32
SM_STRIDE = 33  # matmul-style pad: row stride HALO+1 avoids 32-bank conflicts on row reads
SM_HALO = HALO * SM_STRIDE  # 1056 bytes (was 32*32=1024)
ROW_STEP = TILE  # non-overlapping row tiles; 28 left gaps on rows 0 and N-1


@cuda.jit
def kernel_tile30_halo(padded, N, block_out):
    """Method 5: 30x30 tile, 32x32 halo in sm; 256 threads scan tile + block reduce."""
    tr = cuda.blockIdx.y * ROW_STEP
    tc = cuda.blockIdx.x * TILE
    bdx = cuda.blockDim.x
    tid = cuda.threadIdx.y * bdx + cuda.threadIdx.x
    nt = bdx * cuda.blockDim.y

    sm = cuda.shared.array(SM_HALO, dtype=numba.uint8)
    idx = tid
    while idx < HALO * HALO:
        ly = idx // HALO
        lx = idx % HALO
        sr = tr + ly
        sc = tc + lx
        si = ly * SM_STRIDE + lx
        if sr < N + 2 and sc < N + 2:
            sm[si] = padded[sr, sc]
        else:
            sm[si] = 0
        idx += nt
    cuda.syncthreads()

    local = 0
    if tr < N and tc < N:
        lim_r = TILE
        lim_c = TILE
        if tr + TILE > N:
            lim_r = N - tr
        if tc + TILE > N:
            lim_c = N - tc
        ncells = lim_r * lim_c
        idx = tid
        while idx < ncells:
            a = idx // lim_c
            b = idx % lim_c
            ly = a + 1
            lx = b + 1
            v = sm[ly * SM_STRIDE + lx]
            if v == 1:
                n = (
                    sm[(ly - 1) * SM_STRIDE + lx - 1]
                    + sm[(ly - 1) * SM_STRIDE + lx]
                    + sm[(ly - 1) * SM_STRIDE + lx + 1]
                    + sm[ly * SM_STRIDE + lx - 1]
                    + sm[ly * SM_STRIDE + lx + 1]
                    + sm[(ly + 1) * SM_STRIDE + lx - 1]
                    + sm[(ly + 1) * SM_STRIDE + lx + 1]
                    + sm[(ly + 1) * SM_STRIDE + lx]
                )
                if n == 0:
                    local += 1
            idx += nt

    sm_sum = cuda.shared.array(256, dtype=numba.int32)
    sm_sum[tid] = local
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if tid < stride:
            sm_sum[tid] += sm_sum[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = sm_sum[0]


TILE16 = 16
HALO16 = 18
SM16 = 324  # 18*18 halo; one 16x16 tile == 256 threads (no stride loop)


@cuda.jit
def kernel_tile16_halo(padded, N, block_out):
    """Method 5b: tile == block (16x16); 1 thread/cell; same grid count as method 2."""
    tr = cuda.blockIdx.y * TILE16
    tc = cuda.blockIdx.x * TILE16
    ty = cuda.threadIdx.y
    tx = cuda.threadIdx.x
    tid = ty * TILE16 + tx

    sm = cuda.shared.array(SM16, dtype=numba.uint8)
    idx = tid
    while idx < SM16:
        ly = idx // HALO16
        lx = idx % HALO16
        sr = tr + ly
        sc = tc + lx
        if sr < N + 2 and sc < N + 2:
            sm[idx] = padded[sr, sc]
        else:
            sm[idx] = 0
        idx += 256
    cuda.syncthreads()

    local = 0
    r = tr + ty
    c = tc + tx
    if r < N and c < N:
        ly = ty + 1
        lx = tx + 1
        v = sm[ly * HALO16 + lx]
        if v == 1:
            n = (
                sm[(ly - 1) * HALO16 + lx - 1]
                + sm[(ly - 1) * HALO16 + lx]
                + sm[(ly - 1) * HALO16 + lx + 1]
                + sm[ly * HALO16 + lx - 1]
                + sm[ly * HALO16 + lx + 1]
                + sm[(ly + 1) * HALO16 + lx - 1]
                + sm[(ly + 1) * HALO16 + lx]
                + sm[(ly + 1) * HALO16 + lx + 1]
            )
            if n == 0:
                local = 1

    sm_sum = cuda.shared.array(256, dtype=numba.int32)
    sm_sum[tid] = local
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if tid < stride:
            sm_sum[tid] += sm_sum[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = sm_sum[0]


@cuda.jit
def kernel_tile30_nbrcache(padded, N, block_out):
    """Method 5c: precompute 8-neighbor sum in sm_nbr (software cache). Wins dense, loses sparse."""
    tr = cuda.blockIdx.y * ROW_STEP
    tc = cuda.blockIdx.x * TILE
    bdx = cuda.blockDim.x
    tid = cuda.threadIdx.y * bdx + cuda.threadIdx.x
    nt = bdx * cuda.blockDim.y

    sm = cuda.shared.array(SM_HALO, dtype=numba.uint8)
    sm_nbr = cuda.shared.array(SM_HALO, dtype=numba.uint8)
    idx = tid
    while idx < HALO * HALO:
        ly = idx // HALO
        lx = idx % HALO
        sr = tr + ly
        sc = tc + lx
        si = ly * SM_STRIDE + lx
        if sr < N + 2 and sc < N + 2:
            sm[si] = padded[sr, sc]
        else:
            sm[si] = 0
        idx += nt
    cuda.syncthreads()

    local = 0
    if tr < N and tc < N:
        lim_r = TILE if tr + TILE <= N else N - tr
        lim_c = TILE if tc + TILE <= N else N - tc
        ncells = lim_r * lim_c
        idx = tid
        while idx < ncells:
            a = idx // lim_c
            b = idx % lim_c
            ly = a + 1
            lx = b + 1
            si = ly * SM_STRIDE + lx
            sm_nbr[si] = (
                sm[(ly - 1) * SM_STRIDE + lx - 1]
                + sm[(ly - 1) * SM_STRIDE + lx]
                + sm[(ly - 1) * SM_STRIDE + lx + 1]
                + sm[ly * SM_STRIDE + lx - 1]
                + sm[ly * SM_STRIDE + lx + 1]
                + sm[(ly + 1) * SM_STRIDE + lx - 1]
                + sm[(ly + 1) * SM_STRIDE + lx]
                + sm[(ly + 1) * SM_STRIDE + lx + 1]
            )
            idx += nt
    cuda.syncthreads()

    if tr < N and tc < N:
        lim_r = TILE if tr + TILE <= N else N - tr
        lim_c = TILE if tc + TILE <= N else N - tc
        ncells = lim_r * lim_c
        idx = tid
        while idx < ncells:
            a = idx // lim_c
            b = idx % lim_c
            ly = a + 1
            lx = b + 1
            si = ly * SM_STRIDE + lx
            if sm[si] == 1 and sm_nbr[si] == 0:
                local += 1
            idx += nt

    sm_sum = cuda.shared.array(256, dtype=numba.int32)
    sm_sum[tid] = local
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if tid < stride:
            sm_sum[tid] += sm_sum[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = sm_sum[0]


BLOCK30 = TILE * TILE  # 900 threads; tile == block for method 5d
SM_SUM1024 = 1024  # padded block reduce (900 active lanes)


@cuda.jit
def kernel_tile30_block30(padded, N, block_out):
    """Method 5d: 30x30 tile == 30x30 block; 1 thread/cell; reduce in sm_sum[1024]."""
    tr = cuda.blockIdx.y * ROW_STEP
    tc = cuda.blockIdx.x * TILE
    ty = cuda.threadIdx.y
    tx = cuda.threadIdx.x
    tid = ty * TILE + tx

    sm = cuda.shared.array(SM_HALO, dtype=numba.uint8)
    idx = tid
    while idx < HALO * HALO:
        ly = idx // HALO
        lx = idx % HALO
        sr = tr + ly
        sc = tc + lx
        si = ly * SM_STRIDE + lx
        if sr < N + 2 and sc < N + 2:
            sm[si] = padded[sr, sc]
        else:
            sm[si] = 0
        idx += BLOCK30
    cuda.syncthreads()

    local = 0
    r = tr + ty
    c = tc + tx
    if r < N and c < N:
        ly = ty + 1
        lx = tx + 1
        v = sm[ly * SM_STRIDE + lx]
        if v == 1:
            n = (
                sm[(ly - 1) * SM_STRIDE + lx - 1]
                + sm[(ly - 1) * SM_STRIDE + lx]
                + sm[(ly - 1) * SM_STRIDE + lx + 1]
                + sm[ly * SM_STRIDE + lx - 1]
                + sm[ly * SM_STRIDE + lx + 1]
                + sm[(ly + 1) * SM_STRIDE + lx - 1]
                + sm[(ly + 1) * SM_STRIDE + lx]
                + sm[(ly + 1) * SM_STRIDE + lx + 1]
            )
            if n == 0:
                local = 1

    sm_sum = cuda.shared.array(SM_SUM1024, dtype=numba.int32)
    sm_sum[tid] = local
    cuda.syncthreads()
    stride = 512
    while stride > 0:
        if tid < stride:
            other = tid + stride
            if other < BLOCK30:
                sm_sum[tid] += sm_sum[other]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = sm_sum[0]


BITMASK_WARP = 32
TILE_K3_SM = 25  # (k+2)^2 for k=3
BITMASK_SM_HALO = 36  # (k+2)^2 for k=4


@cuda.jit(device=True)
def _dev_count_tile_stencil_at(sm, base, k, halo):
    local = 0
    for i in range(1, k + 1):
        for j in range(1, k + 1):
            si = base + i * halo + j
            if sm[si] != 1:
                continue
            ok = True
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    if di == 0 and dj == 0:
                        continue
                    if sm[base + (i + di) * halo + (j + dj)] != 0:
                        ok = False
                        break
                if not ok:
                    break
            if ok:
                local += 1
    return local


@cuda.jit(device=True)
def _dev_count_tile_stencil(sm, k, halo):
    return _dev_count_tile_stencil_at(sm, 0, k, halo)


@cuda.jit
def kernel_lut_k3_warp(padded, N, k, halo, lut, block_out):
    """6_lut_k3: warp/tile, cooperative load, lut[bits]."""
    tr = cuda.blockIdx.y * k
    tc = cuda.blockIdx.x * k
    tid = cuda.threadIdx.x
    sm = cuda.shared.array(TILE_K3_SM, dtype=numba.uint8)
    nh = halo * halo
    if tid < nh:
        i = tid // halo
        j = tid % halo
        sr = tr + i
        sc = tc + j
        v = 0
        if sr < padded.shape[0] and sc < padded.shape[1]:
            v = padded[sr, sc]
        sm[tid] = v
    cuda.syncthreads()
    if tid == 0:
        bits = 0
        for idx in range(nh):
            if sm[idx]:
                bits |= 1 << idx
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = lut[bits]


@cuda.jit
def kernel_tile_k3_mtwp(padded, N, k, halo, tiles_per_warp, block_out):
    """6a: k=3 stencil; one warp processes tiles_per_warp tiles serially."""
    tid = cuda.threadIdx.x
    bi = cuda.blockIdx.x
    nt = (N + k - 1) // k
    ntiles = nt * nt
    nh = halo * halo

    sm = cuda.shared.array(TILE_K3_SM, dtype=numba.uint8)
    tile_base = bi * tiles_per_warp
    for t in range(tiles_per_warp):
        tile_id = tile_base + t
        if tile_id >= ntiles:
            break
        tr = (tile_id // nt) * k
        tc = (tile_id % nt) * k
        idx = tid
        while idx < nh:
            i = idx // halo
            j = idx % halo
            sr = tr + i
            sc = tc + j
            v = 0
            if sr < padded.shape[0] and sc < padded.shape[1]:
                v = padded[sr, sc]
            sm[idx] = v
            idx += BITMASK_WARP
        cuda.syncthreads()
        if tid == 0:
            block_out[tile_id] = _dev_count_tile_stencil(sm, k, halo)
        cuda.syncthreads()


@cuda.jit
def kernel_tile_k4_warp(padded, N, k, halo, block_out):
    """Method 6: warp/tile k=4; cooperative halo load; stencil count in shared."""
    tr = cuda.blockIdx.y * k
    tc = cuda.blockIdx.x * k
    tid = cuda.threadIdx.x

    sm = cuda.shared.array(BITMASK_SM_HALO, dtype=numba.uint8)
    nh = halo * halo
    idx = tid
    while idx < nh:
        i = idx // halo
        j = idx % halo
        sr = tr + i
        sc = tc + j
        v = 0
        if sr < padded.shape[0] and sc < padded.shape[1]:
            v = padded[sr, sc]
        sm[idx] = v
        idx += BITMASK_WARP
    cuda.syncthreads()

    if tid == 0:
        bi = cuda.blockIdx.y * cuda.gridDim.x + cuda.blockIdx.x
        block_out[bi] = _dev_count_tile_stencil(sm, k, halo)


print("Kernels defined.")
print("Note: shared array sizes must be literals in @cuda.jit (numba_cuda).")

## Host wrappers

In [ ]:
def _grid_2d(N):
    bx = (N + TPB - 1) // TPB
    return (bx, bx)


def _blocks_1d(n, tpb=256):
    return ((n + tpb - 1) // tpb,), (tpb,)


def run_interview_if_atomic(d_grid, N):
    out = cuda.to_device(np.zeros(1, dtype=np.int32))
    kernel_interview_if_atomic[_grid_2d(N), (TPB, TPB)](d_grid, N, out)
    cuda.synchronize()
    return int(out.copy_to_host()[0])


def run_divergent_early_return(d_grid, N):
    out = cuda.to_device(np.zeros(1, dtype=np.int32))
    kernel_divergent[_grid_2d(N), (TPB, TPB)](d_grid, N, out)
    cuda.synchronize()
    return int(out.copy_to_host()[0])


def run_uniform(d_padded, N):
    bx = (N + TPB - 1) // TPB
    nblocks = bx * bx
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_uniform_padded[(bx, bx), (TPB, TPB)](d_padded, N, block_out)
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def run_compact(d_padded, h_grid, N):
    """Compaction via CuPy flatnonzero (GPU), then sparse CUDA kernel."""
    g = cp.asarray(h_grid)
    positions = cp.flatnonzero(g).astype(cp.int32)
    num_ones = int(positions.size)
    d_pos = cuda.as_cuda_array(positions)
    out = cuda.to_device(np.zeros(1, dtype=np.int32))
    if num_ones > 0:
        kernel_compact_check[_blocks_1d(num_ones)](d_padded, N, d_pos, num_ones, out)
    cuda.synchronize()
    return int(out.copy_to_host()[0])


def run_ballot(d_padded, N):
    bx = (N + TPB - 1) // TPB
    nblocks = bx * bx
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_ballot_padded[(bx, bx), (TPB, TPB)](d_padded, N, block_out)
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def _grid_tile30(N):
    bx = (N + TILE - 1) // TILE
    by = (N + ROW_STEP - 1) // ROW_STEP
    return (bx, by)


def run_tile30_halo(d_padded, N):
    bx, by = _grid_tile30(N)
    nblocks = bx * by
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_tile30_halo[(bx, by), (TPB, TPB)](d_padded, N, block_out)
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def _grid_tile16(N):
    bx = (N + TILE16 - 1) // TILE16
    return (bx, bx)


def run_tile16_halo(d_padded, N):
    bx, by = _grid_tile16(N)
    nblocks = bx * by
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_tile16_halo[(bx, by), (TPB, TPB)](d_padded, N, block_out)
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def run_tile30_nbrcache(d_padded, N):
    bx, by = _grid_tile30(N)
    nblocks = bx * by
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_tile30_nbrcache[(bx, by), (TPB, TPB)](d_padded, N, block_out)
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def run_tile30_block30(d_padded, N):
    bx, by = _grid_tile30(N)
    nblocks = bx * by
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_tile30_block30[(bx, by), (TILE, TILE)](d_padded, N, block_out)
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def _grid_bit_tile(N):
    k = BIT_TILE_K
    bx = (N + k - 1) // k
    return (bx, bx)


def _grid_tile_k3_blocks(N, tiles_per_block):
    k = TILE_K3
    nt = (N + k - 1) // k
    ntiles = nt * nt
    nblocks = (ntiles + tiles_per_block - 1) // tiles_per_block
    return nt, ntiles, nblocks


_LUT_K3_DEV = None


def _lut_k3_dev():
    global _LUT_K3_DEV
    if _LUT_K3_DEV is None:
        _LUT_K3_DEV = cuda.to_device(TILE_K3_LUT)
    return _LUT_K3_DEV


def _grid_lut_k3(N):
    k = TILE_K3
    bx = (N + k - 1) // k
    return (bx, bx)


def run_lut_k3_warp(d_padded, N):
    k = TILE_K3
    halo = TILE_K3_HALO
    bx, by = _grid_lut_k3(N)
    nblocks = bx * by
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_lut_k3_warp[(bx, by), (BITMASK_WARP, 1)](
        d_padded, N, k, halo, _lut_k3_dev(), block_out
    )
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def run_tile_k3_mtwp(d_padded, N):
    k = TILE_K3
    halo = TILE_K3_HALO
    nt, ntiles, nblocks = _grid_tile_k3_blocks(N, TILES_PER_WARP)
    block_out = cuda.to_device(np.zeros(ntiles, dtype=np.int32))
    kernel_tile_k3_mtwp[(nblocks,), (BITMASK_WARP,)](
        d_padded, N, k, halo, TILES_PER_WARP, block_out
    )
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


def run_tile_k4_warp(d_padded, N):
    k = BIT_TILE_K
    halo = BIT_TILE_HALO
    bx, by = _grid_bit_tile(N)
    nblocks = bx * by
    block_out = cuda.to_device(np.zeros(nblocks, dtype=np.int32))
    kernel_tile_k4_warp[(bx, by), (BITMASK_WARP, 1)](
        d_padded, N, k, halo, block_out
    )
    cuda.synchronize()
    return int(block_out.copy_to_host().sum())


METHODS = {
    "1_interview_if_atomic": run_interview_if_atomic,
    "1b_divergent_early_return": run_divergent_early_return,
    "2_uniform_block_reduce": run_uniform,
    "3_compaction": run_compact,
    "4_warp_ballot": run_ballot,
    "5_tile30_halo": run_tile30_halo,
    "5b_tile16_halo": run_tile16_halo,
    "5c_tile30_nbrcache": run_tile30_nbrcache,
    "5d_tile30_block30": run_tile30_block30,
    "6_lut_k3_warp": run_lut_k3_warp,
}
GRID_ATOMIC_RUNS = (run_interview_if_atomic, run_divergent_early_return)
TILE_HALO_RUNS = (
    run_tile30_halo,
    run_tile16_halo,
    run_tile30_nbrcache,
    run_tile30_block30,
)
BITMASK_RUNS = (run_lut_k3_warp,)
SIM_SKIP = (run_interview_if_atomic, run_divergent_early_return, run_compact)


def _using_cuda_simulator():
    try:
        return bool(CUDA_SIMULATOR)
    except NameError:
        import os

        return os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"


# Launch config (block size / grid) — N-dependent grid shown as formula
METHOD_LAUNCH = {
    "0_cpu_reference": "CPU nested loops",
    "1_interview_if_atomic": f"grid ({{bx}},{{bx}}) bx=ceil(N/{TPB}), block ({TPB},{TPB}) = 256 thr",
    "1b_divergent_early_return": f"grid ({{bx}},{{bx}}) bx=ceil(N/{TPB}), block ({TPB},{TPB}) = 256 thr",
    "2_uniform_block_reduce": f"grid ({{bx}},{{bx}}) bx=ceil(N/{TPB}), block ({TPB},{TPB}) = 256 thr",
    "3_compaction": "CuPy flatnonzero + grid (ceil(num_ones/256),) block (256,) 1D",
    "4_warp_ballot": f"grid ({{bx}},{{bx}}) bx=ceil(N/{TPB}), block ({TPB},{TPB}) = 256 thr",
    "5_tile30_halo": f"grid ({{bx}},{{by}}) bx=ceil(N/{TILE}), by=ceil(N/{TILE}), block ({TPB},{TPB}) = 256 thr",
    "5b_tile16_halo": f"grid ({{bx}},{{bx}}) bx=ceil(N/{TILE16}), block ({TPB},{TPB}) = 256 thr",
    "5c_tile30_nbrcache": f"grid ({{bx}},{{by}}) bx=ceil(N/{TILE}), by=ceil(N/{TILE}), block ({TPB},{TPB}) = 256 thr",
    "5d_tile30_block30": f"grid ({{bx}},{{by}}) bx=ceil(N/{TILE}), block ({TILE},{TILE}) = {TILE*TILE} thr (tile==block)",
    "6_lut_k3_warp": f"grid ({{bx}},{{bx}}) bx=ceil(N/3), block ({BITMASK_WARP},1) k=3 LUT",
}


def launch_detail(method_name, N):
    """Concrete grid/block for this N (for benchmark tables)."""
    bx = (N + TPB - 1) // TPB
    bx30 = (N + TILE - 1) // TILE
    by30 = (N + ROW_STEP - 1) // ROW_STEP
    bx16t = (N + TILE16 - 1) // TILE16
    b16 = f"grid ({bx},{bx}) block ({TPB},{TPB})"
    t30 = f"grid ({bx30},{by30}) block ({TPB},{TPB})"
    return {
        "0_cpu_reference": "CPU",
        "1_interview_if_atomic": b16,
        "1b_divergent_early_return": b16,
        "2_uniform_block_reduce": b16,
        "3_compaction": "flatnonzero + grid (ceil(num_ones/256),) block (256,)",
        "4_warp_ballot": b16,
        "5_tile30_halo": t30,
        "5b_tile16_halo": f"grid ({bx16t},{bx16t}) block ({TPB},{TPB})",
        "5c_tile30_nbrcache": t30,
        "5d_tile30_block30": f"grid ({bx30},{by30}) block ({TILE},{TILE})",
        "6_lut_k3_warp": (
            f"grid ({(N + 2) // 3},{(N + 2) // 3}) block ({BITMASK_WARP},1) k=3 LUT"
        ),
    }.get(method_name, "?")


WARMUP = 2
REPEATS = 10


def benchmark_method(fn, h_grid, d_grid, d_padded, N, repeats=REPEATS):
    for _ in range(WARMUP):
        if fn in GRID_ATOMIC_RUNS:
            fn(d_grid, N)
        elif fn in TILE_HALO_RUNS:
            fn(d_padded, N)
        elif fn in BITMASK_RUNS:
            fn(d_padded, N)
        elif fn is run_compact:
            fn(d_padded, h_grid, N)
        else:
            fn(d_padded, N)
    cuda.synchronize()

    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        if fn in GRID_ATOMIC_RUNS:
            fn(d_grid, N)
        elif fn in TILE_HALO_RUNS:
            fn(d_padded, N)
        elif fn in BITMASK_RUNS:
            fn(d_padded, N)
        elif fn is run_compact:
            fn(d_padded, h_grid, N)
        else:
            fn(d_padded, N)
        cuda.synchronize()
        times.append(time.perf_counter() - t0)
    times = np.array(times) * 1e3
    return {
        "median_ms": float(np.median(times)),
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
    }

## Correctness check

In [ ]:
def check_cpu_tile_k3(N, density=0.08, seed=42):
    h = make_grid(N, density, seed)
    p = pad_grid(h)
    ref = count_isolated_cpu(h)
    got = count_isolated_tile_cpu(p, N, TILE_K3)
    ok = got == ref
    print(f"  CPU tile k=3 N={N} ref={ref} got={got} [{'OK' if ok else 'MISMATCH'}]")
    return ok


def check_all_methods(N, density=0.08, seed=42):
    h = make_grid(N, density, seed)
    ref = count_isolated_cpu(h)
    d_g = cuda.to_device(h)
    p = pad_grid(h)
    d_p = cuda.to_device(p)

    results = {}
    for name, fn in METHODS.items():
        if fn in GRID_ATOMIC_RUNS:
            results[name] = fn(d_g, N)
        elif fn is run_compact:
            results[name] = fn(d_p, h, N)
        elif fn in BITMASK_RUNS:
            results[name] = fn(d_p, N)
        else:
            results[name] = fn(d_p, N)

    ok = all(v == ref for v in results.values())
    print(f"N={N} density={density} islands(ref)={ref} OK={ok}")
    for k, v in results.items():
        mark = "OK" if v == ref else "MISMATCH"
        print(f"  {k}: {v} [{mark}]")
    return ok


if _using_cuda_simulator():
    print("CUDA simulator: skipping all GPU kernels (device writes broken in numba 0.60).")
    print("CPU-only checks:")
    cpu_ok = all(check_cpu_tile_k3(N) for N in (16, 64, 256))
    # spot-check LUT vs bit formula
    for bits in (0, 1, 0x15555, (1 << TILE_K3_LUT_BITS) - 1):
        assert TILE_K3_LUT[bits] == _count_from_bits(bits, TILE_K3, TILE_K3_HALO)
    print("  LUT spot-check OK")
    print(f"\nCPU tile path OK={cpu_ok}. Use cell 1 + real GPU for full METHODS check and benchmark.")
else:
    for N in (64, 256, 1024):
        check_all_methods(N)
    print("\nIf all OK, run benchmark below.")

## Benchmark: N = 1024 and N = 2048

**Storage:** always a **dense** `uint8` `N×N` array on GPU (and padded `(N+2)²` where needed). We do *not* use CSR/COO sparse matrix formats.

**Sparsity:** `density` = probability each cell is `1` (Bernoulli). Low density → few `1`s; high density → most cells `1`. Kernels 1/2/4/5 still launch over the **full grid**; method 3 compacts only the `1` positions but starts from the dense array.

Three regimes in `DENSITY_SCENARIOS`: **sparse** (`0.01`), **medium** (`0.10`), **dense** (`0.90`). Edit `SIZES` / scenarios as needed.

**Timing scope:** each `median_ms` is **kernel + host sync only** (per matrix). `6_lut_k3_warp` LUT build/upload happens **before** the timed loop.

In [ ]:
SIZES = [1024, 2048]
DENSITY_SCENARIOS = [
    (0.01, "sparse"),
    (0.10, "medium"),
    (0.90, "dense"),
]
SEED = 7

rows = []

if _using_cuda_simulator():
    print(
        "Benchmark skipped (CUDA simulator / cell 1b).\n"
        "  -> Runtime -> T4 GPU -> Restart session -> run cell 1, then re-run benchmark."
    )
else:
    # LUT on device before timed runs (offline build excluded from benchmark)
    _lut_k3_dev()

    for density, label in DENSITY_SCENARIOS:
        for N in SIZES:
            print(f"\n=== N={N}  density={density} ({label}) ===")
            h = make_grid(N, density, SEED + N + int(density * 1000))
            ref = count_isolated_cpu(h)
            d_g = cuda.to_device(h)
            d_p = cuda.to_device(pad_grid(h))
            n_ones = int(h.sum())
            print(f"  ones: {n_ones}  isolated (ref): {ref}")

            for name, fn in METHODS.items():
                if fn in GRID_ATOMIC_RUNS:
                    val = fn(d_g, N)
                elif fn in TILE_HALO_RUNS:
                    val = fn(d_p, N)
                elif fn in BITMASK_RUNS:
                    val = fn(d_p, N)
                elif fn is run_compact:
                    val = fn(d_p, h, N)
                else:
                    val = fn(d_p, N)
                assert val == ref, (label, name, val, ref)
                stats = benchmark_method(fn, h, d_g, d_p, N)
                rows.append({
                    "N": N,
                    "density": density,
                    "regime": label,
                    "method": name,
                    "launch": launch_detail(name, N),
                    "count": val,
                    "n_ones": n_ones,
                    **stats,
                })
                print(f"  {name:28s}  median {stats['median_ms']:7.2f} ms  (std {stats['std_ms']:.2f})")

            t0 = time.perf_counter()
            count_isolated_cpu(h)
            cpu_ms = (time.perf_counter() - t0) * 1e3
            rows.append({
                "N": N,
                "density": density,
                "regime": label,
                "method": "0_cpu_reference",
                "launch": launch_detail("0_cpu_reference", N),
                "count": ref,
                "n_ones": n_ones,
                "median_ms": cpu_ms,
                "mean_ms": cpu_ms,
                "std_ms": 0.0,
            })
            print(f"  CPU reference: {cpu_ms:.1f} ms")

In [ ]:
import pandas as pd

if not rows:
    print("No benchmark rows — run benchmark on a real GPU (cell 1, not 1b).")
else:
    df = pd.DataFrame(rows)

    print("--- Launch config (block / grid) per method ---")
    launch_ref = pd.DataFrame(
        [{"method": k, "launch_formula": v} for k, v in METHOD_LAUNCH.items()]
    ).set_index("method")
    display(launch_ref)
    for N in SIZES:
        print(f"\n--- Launch at N={N} (concrete grid) ---")
        display(
            pd.DataFrame(
                [{"method": m, "launch": launch_detail(m, N)} for m in list(METHOD_LAUNCH.keys())]
            ).set_index("method")
        )

    for label in df["regime"].unique():
        sub = df[df["regime"] == label]
        print(f"\n--- median ms ({label}) ---")
        display(sub.pivot_table(index="N", columns="method", values="median_ms").sort_index())
        print(f"--- launch @ each N ({label}) ---")
        display(
            sub.groupby(["N", "method"], as_index=False)["launch"]
            .first()
            .pivot(index="N", columns="method", values="launch")
            .sort_index()
        )

    print("\n--- 1 vs 1b side by side (same logic, different branch style) ---")
    m1, m1b = "1_interview_if_atomic", "1b_divergent_early_return"
    cmp = df[df["method"].isin([m1, m1b])].pivot_table(
        index=["regime", "N"], columns="method", values="median_ms"
    )
    cmp["1_over_1b"] = cmp[m1] / cmp[m1b]
    cmp["1b_over_1"] = cmp[m1b] / cmp[m1]
    cmp["delta_ms_1b_minus_1"] = cmp[m1b] - cmp[m1]
    display(
        cmp[[m1, m1b, "1_over_1b", "1b_over_1", "delta_ms_1b_minus_1"]].round(4)
    )
    print("> 1_over_1b > 1 means interview (1) is faster; ~1.0 => same efficiency class.")

    for label in df["regime"].unique():
        for N in SIZES:
            sub = df[(df["regime"] == label) & (df["N"] == N) & (df["method"] != "0_cpu_reference")]
            base_rows = sub.loc[sub["method"] == m1, "median_ms"]
            t1b_rows = sub.loc[sub["method"] == m1b, "median_ms"]
            if base_rows.empty:
                print(f"\nN={N} {label}: skip speedup")
                continue
            base = float(base_rows.iloc[0])
            print(f"\nN={N}  regime={label}")
            if not t1b_rows.empty:
                t1b = float(t1b_rows.iloc[0])
                print(f"  1 vs 1b:  interview {base:.3f} ms  |  early_return {t1b:.3f} ms  |  1/1b={base/t1b:.2f}x  1b/1={t1b/base:.2f}x")
            print(f"  speedup vs interview (1):")
            for m in sorted(sub["method"].unique()):
                if m == m1:
                    continue
                t = float(sub.loc[sub["method"] == m, "median_ms"].iloc[0])
                print(f"    {m:28s}  {base / t:.2f}x")

In [ ]:
# Optional plot (Colab) — one subplot per density; fixed color per method
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

if not rows:
    print("Plot skipped (no benchmark data).")
else:
    gpu_df = df[df["method"] != "0_cpu_reference"]
    methods = list(METHODS.keys())
    n_methods = len(methods)
    palette = plt.cm.tab10(np.linspace(0, 1, max(n_methods, 10)))
    method_colors = {m: palette[k] for k, m in enumerate(methods)}

    bar_w = 0.09
    group_gap = 0.55  # space between N=1024 and N=2048 groups
    group_pitch = n_methods * bar_w + group_gap
    group_center = (n_methods - 1) * bar_w / 2

    fig, axes = plt.subplots(
        len(DENSITY_SCENARIOS), 1,
        figsize=(11, 3.6 * len(DENSITY_SCENARIOS)),
        sharex=False,
    )
    if len(DENSITY_SCENARIOS) == 1:
        axes = [axes]
    for ax, (density, label) in zip(axes, DENSITY_SCENARIOS):
        part = gpu_df[gpu_df["regime"] == label]
        for i, N in enumerate(SIZES):
            x0 = i * group_pitch
            for k, m in enumerate(methods):
                y = part[(part["N"] == N) & (part["method"] == m)]["median_ms"].iloc[0]
                ax.bar(x0 + k * bar_w, y, width=bar_w, color=method_colors[m])
        tick_x = [i * group_pitch + group_center for i in range(len(SIZES))]
        ax.set_xticks(tick_x)
        ax.set_xticklabels([str(n) for n in SIZES], fontsize=11)
        ax.set_xlim(-bar_w, len(SIZES) * group_pitch - group_gap + n_methods * bar_w)
        ax.set_ylabel("median ms")
        ax.set_title(f"{label} (density={density})")
        ax.grid(axis="y", alpha=0.25)
    axes[-1].set_xlabel("matrix size N")
    legend_patches = [Patch(facecolor=method_colors[m], label=m) for m in methods]
    fig.legend(
        handles=legend_patches,
        fontsize=6,
        ncol=2,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.01),
    )
    fig.subplots_adjust(top=0.90, bottom=0.14)
    plt.show()

## Tile halo bottlenecks (method 5)

**Not bad L2/L1 "cache misses" on global** — each tile loads the halo into **shared memory (LDS) once**; the hot path reads `sm`, not `padded`.

### Matmul lesson: banks, broadcast, padding

CUDA tiled **matrix multiply** uses `__shared__ tile[TILE][TILE + 1]` so a warp reading **across columns** of a row does not hit **32-way bank conflicts** (stride-32 rows map many lanes to the same bank).

- **Broadcast:** if all 32 lanes read the **same** shared address (e.g. one matrix element reused down a K-dimension step), the hardware **broadcasts** one load to the warp.
- **Our stencil:** lanes read **different columns** (`lx = thread + 1`) on rows `ly-1, ly, ly+1` — broadcast helps less; **bank conflicts** on row walks hurt more when row stride = **32 bytes** (`uint8` halo width 32).
- **Fix (in `5_tile30_halo` / `5c`):** `SM_STRIDE = HALO + 1` (33), index `ly * SM_STRIDE + lx` — same idea as matmul `TILE+1` padding.
- **`5b_tile16_halo`:** row stride 18 — already not a multiple of 32 banks for 4-byte bank width.
- **Next step (HIP/CUDA C++):** pack rows as `uint32` / `float4` so each lane reads **4 bytes** (one bank), or use **`__shfl_sync`** to pass left/right neighbors without re-reading shared.

| Cost | `5_tile30_halo` today |
|------|------------------------|
| Extra blocks vs method 2 | ~35² vs 64² at N=1024 — fewer blocks but **~3–4 cells/thread** (stride loop) |
| `idx // lim_c`, `idx % lim_c` | Integer div/mod **per cell** in the stride loop |
| Redundant `sm` reads | Adjacent cells re-read overlapping 3×3 neighborhoods — only painful when **many** cells are `1` (dense) |
| `syncthreads` | After halo load + after block reduce (same as method 2) |
| `5c` nbr cache | **2×** shared traffic (fill `sm_nbr` for **all** cells) — wins dense, loses sparse |

**Software cache idea:** store **neighbor-sum** (0–8) per cell in shared once, then `isolated = (v==1 && nbr==0)`. That removes duplicate neighbor work but **always** pays for empty cells. Current code only sums neighbors when `v==1` (lazy) — optimal for **sparse** content.

**Best structural fix:** `5b_tile16_halo` — **16×16 tile = 16×16 threads**, one cell per lane, grid matches method 2.

## Notes for the interview

- **Dense layout vs sparse content:** matrices are always dense `N×N`; benchmark **sparse** (`0.01`), **medium** (`0.10`), **dense** (`0.9`).
- **Method 1** uses divergent early `return` - often wins when few cells are `1` (sparse); less clear when most cells are `1` (dense).
- **Method 2** is what you should defend verbally: uniform path + block reduction (AMD: same idea with wave64 + LDS).
- **Method 3** avoids visiting `0` cells in the *second* kernel; `flatnonzero` cost matters - wins when density is low.
- **Method 4** does **per-warp reduction** (lane 0 sums 32 lanes). On Colab `numba_cuda` has no `cuda.vote`; in CUDA C++/HIP use `__ballot_sync` instead.
- Colab = **NVIDIA warp 32**; on AMD CDNA use **wave 64** - ballot width changes, idea does not.
- **Method 5 / 5b / 5c** (tile + LDS): `SM_STRIDE = HALO+1` padded shared layout (matmul-style); see *Tile halo bottlenecks*.
- **5b_tile16_halo**: tile size = **16x16** = thread block (no stride loop, same block count as method 2).
- **5c_tile30_nbrcache**: **software cache** — precompute 8-neighbor sum in `sm_nbr` for every cell; better when many `1`s, worse when few.
- **5d_tile30_block30**: **30×30 tile = 30×30 block** (900 threads, 1 thread/cell); same grid as method 5, no stride scan loop. May hit `LAUNCH_OUT_OF_RESOURCES` on Colab T4 (register/shared pressure).
- **6_lut_k3_warp**: k=3 **2^25 LUT** (~32 MB), warp/tile — best tile approach (~1.7 ms sparse @ N=1024); still behind method 1.

To disable methods temporarily, edit `METHODS` dict in the host wrappers cell.